**From a series of application of modern interpolation methods for economics: written by [Mahdi E Kahou](https://mekahou.github.io/)**

# Spectral bias animation

This notebook trains the same network as `spectral_bias_DL.ipynb` and records the fit during training, then animates the solution alongside its Fourier transform so the low-to-high ordering is visible frame by frame.

In [2]:
## Importing packages
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import matplotlib.pyplot as plt
from matplotlib import cm
import os
import imageio.v2 as imageio  # use v2 API to avoid warning

In [3]:
fontsize= 14
ticksize = 14
figsize = (15, 6.5)
params_fig = {'font.family':'serif',
    "figure.figsize":figsize,
    'figure.dpi': 80,
    'figure.edgecolor': 'k',
    'font.size': fontsize,
    'axes.labelsize': fontsize,
    'axes.titlesize': fontsize,
    'xtick.labelsize': ticksize,
    'ytick.labelsize': ticksize
}
plt.rcParams.update(params_fig)

In [4]:
## Multi-frequency dgp on [0, 8 pi]
X_min, X_max = 0.0, 8 * np.pi

def dgp(x):
    return (np.sin(x) + np.sin(5 * x) + np.sin(10 * x) + np.sin(20 * x)) / 4.0

def normalize(x):
    # map [X_min, X_max] -> [-1, 1] before feeding the network
    return 2.0 * (x - X_min) / (X_max - X_min) - 1.0

In [5]:
## Training data (fixed uniform grid)
n_train = 1000
x_train_np = np.linspace(X_min, X_max, n_train)
y_train_np = dgp(x_train_np)

In [6]:
## Test data (dense grid for plotting / FFT)
n_eval = 8192
x_test_np = np.linspace(X_min, X_max, n_eval)
y_true_np = dgp(x_test_np)

In [7]:
class NN(nn.Module):
    def __init__(self,
                 input_dim = 1,
                 dim_hidden=128,
                 layers=3,
                 hidden_bias=True,
                 hidden_activation=nn.Tanh,
                 seed=123):
        super().__init__()
        self.input_dim = input_dim
        self.dim_hidden = dim_hidden
        self.layers = layers
        self.hidden_bias = hidden_bias
        self.hidden_activation = hidden_activation
        self.seed = seed

        # Set seed if provided
        if self.seed is not None:
            torch.manual_seed(self.seed)
        
        module = []
        
        # First layer
        module.append(nn.Linear(self.input_dim, self.dim_hidden, bias=self.hidden_bias))
        module.append(self.hidden_activation())

        # Additional hidden layers
        for _ in range(self.layers - 1):
            module.append(nn.Linear(self.dim_hidden, self.dim_hidden, bias=self.hidden_bias))
            module.append(self.hidden_activation())

        module.append(nn.Linear(self.dim_hidden, 1))

        self.q = nn.Sequential(*module)

    def forward(self, x):
        return self.q(x)

In [8]:
# the network sees the normalized input, the target is unchanged
x_train = torch.from_numpy(normalize(x_train_np)).float().unsqueeze(dim = 1)
y_train = torch.from_numpy(y_train_np).float().unsqueeze(dim = 1)

train_dataset = TensorDataset(x_train, y_train)
batch_size = len(x_train)
train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle = False)

x_test = torch.from_numpy(normalize(x_test_np)).float().unsqueeze(dim = 1)

In [9]:
num_epochs = 20001
print_freq = 1000
# frame schedule: dense early (low frequencies move fast), sparse later
snap_early_every, snap_late_every, snap_switch = 25, 250, 500

In [10]:
model = NN(dim_hidden=64, layers=3)  # 3 hidden layers of 64, tanh

In [11]:
optimizer = optim.Adam(model.parameters(), lr = 2e-3)  # constant learning rate
criterion = nn.MSELoss()

In [12]:
# saving the results for the animation
u_history = []      # network fit on the eval grid at saved epochs
loss_history = []   # training loss at saved epochs
epoch_history = []  # the epoch index of each saved frame

In [13]:
for epoch in range(num_epochs):
    for x, y in train_loader:
        optimizer.zero_grad()
        y_hat = model(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

    step = snap_late_every if epoch >= snap_switch else snap_early_every
    if epoch % step == 0:
        model.eval()
        with torch.no_grad():
            u_epoch = model(x_test).reshape(-1).cpu().numpy()
        model.train()
        u_history.append(u_epoch)
        loss_history.append(float(loss.item()))
        epoch_history.append(epoch)

    if epoch % print_freq == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.8f}")

Epoch [1/20001], Loss: 0.12598500
Epoch [1001/20001], Loss: 0.09201234
Epoch [2001/20001], Loss: 0.09177566
Epoch [3001/20001], Loss: 0.09131358
Epoch [4001/20001], Loss: 0.09084876
Epoch [5001/20001], Loss: 0.08608028
Epoch [6001/20001], Loss: 0.08370100
Epoch [7001/20001], Loss: 0.07591437
Epoch [8001/20001], Loss: 0.07299262
Epoch [9001/20001], Loss: 0.06348579
Epoch [10001/20001], Loss: 0.05444894
Epoch [11001/20001], Loss: 0.04770096
Epoch [12001/20001], Loss: 0.04343921
Epoch [13001/20001], Loss: 0.04172817
Epoch [14001/20001], Loss: 0.03982293
Epoch [15001/20001], Loss: 0.03844614
Epoch [16001/20001], Loss: 0.03700054
Epoch [17001/20001], Loss: 0.03491429
Epoch [18001/20001], Loss: 0.03184177
Epoch [19001/20001], Loss: 0.02871254
Epoch [20001/20001], Loss: 0.02523525


In [14]:
## Fourier transform (real signal -> one-sided, angular frequency)
dx = x_test_np[1] - x_test_np[0]
omega = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(n_eval, d=dx))  # angular
pos = omega >= 0                           # real signal: one-sided spectrum
omega = omega[pos]

def spectrum(u_vals):
    Fu = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(u_vals))) * dx
    return np.abs(Fu)[pos]

ft_true = spectrum(y_true_np)

## Creating the animation

In [17]:
OMEGA_LIM = 25.0
dgp_label = r"$k(x) = \frac{1}{4}\left(\sin x + \sin 5x + \sin 10x + \sin 20x\right),\quad x \in [0, 8\pi]$"

filenames = []
for i in range(len(epoch_history)):
    u = u_history[i]
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 6.5))

    axL.plot(x_test_np, u, color="red", lw=1.5, label="neural net")
    axL.plot(x_test_np, y_true_np, color="blue", lw=1.0, ls="--", alpha=0.4, label="Training data")
    axL.set_xlim(X_min, X_max)
    axL.set_ylim(-2.0, 2.0)
    axL.set_xlabel("x")
    axL.set_ylabel("k(x)")
    axL.set_title("solution")
    axL.legend(loc="upper right")

    axR.plot(omega, spectrum(u), color="red", lw=1.5, label="neural net")
    axR.plot(omega, ft_true, color="blue", lw=1.0, ls="--", alpha=0.4, label="True function")
    axR.set_xlim(0, OMEGA_LIM)
    axR.set_ylim(-0.3, ft_true.max() * 1.2)
    axR.set_xlabel(r"$\omega$")
    axR.set_ylabel(r"$|\hat{k}(\omega)|$")
    axR.set_title("Fourier transform")
    axR.legend(loc="upper right")

    fig.suptitle(f"epoch {epoch_history[i]}    loss {loss_history[i]:.2e}", fontsize=13)
    fig.text(0.5, 0.02, dgp_label, ha="center", va="bottom", fontsize=18)
    fig.subplots_adjust(bottom=0.18)

    filename = f'frame_{i:03d}.png'
    filenames.append(filename)
    fig.savefig(filename)
    plt.close(fig)

# build mp4 (H.264; crf keeps the file small, like render.py)
with imageio.get_writer('spectral_bias.mp4', fps=8, codec='libx264',
                        macro_block_size=None,
                        output_params=['-crf', '30', '-preset', 'medium']) as writer:
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

# Remove files
for filename in set(filenames):
    os.remove(filename)